# Backstage als Container-Image vorbereiten

Minimaler Ablauf für eine lokale `mybackstage`-Installation ohne Kubernetes.

## 1. In das Backstage-Projekt wechseln

Alle folgenden Befehle werden im Stammverzeichnis von `mybackstage` ausgeführt.

In [ ]:
cd ~/mybackstage

## 2. Produktionskonfiguration erstellen

Backstage muss im Container auf allen Netzwerkinterfaces lauschen. Frontend und Backend werden gemeinsam über Port `7007` bereitgestellt.

In [ ]:
%%bash
cat > app-config.production.yaml <<'EOF'
app:
  title: Backstage Produktion
  baseUrl: http://192.168.1.104:7007

backend:
  baseUrl: http://192.168.1.104:7007
  listen:
    host: 0.0.0.0
    port: 7007
  cors:
    origin: http://192.168.1.104:7007
    methods: [GET, HEAD, PATCH, POST, PUT, DELETE]
    credentials: true
EOF

## 3. Umgebungsvariablen vorbereiten

Secrets werden beim Containerstart übergeben und nicht in das Image eingebaut. Nicht benötigte Variablen können entfernt werden.

In [ ]:
%%bash
cat > .env.production <<'EOF'
MICROSOFT_CLIENT_ID=deine-client-id
MICROSOFT_CLIENT_SECRET=dein-client-secret
AZURE_TENANT_ID=deine-tenant-id
GITHUB_TOKEN=dein-github-token
GOOGLE_CLIENT_ID=deine-google-client-id
GOOGLE_CLIENT_SECRET=dein-google-client-secret
EOF

chmod 600 .env.production

## 4. Secrets vom Build ausschliessen

Die Datei mit den Zugangsdaten darf weder in Git noch in den Docker-Build-Kontext gelangen.

In [ ]:
%%bash
grep -qxF '.env.production' .gitignore 2>/dev/null || echo '.env.production' >> .gitignore
grep -qxF '.env.production' .dockerignore 2>/dev/null || echo '.env.production' >> .dockerignore

## 5. Backstage für Produktion bauen

Zuerst werden die Abhängigkeiten installiert, danach die Typen und das Backend-Bundle erzeugt. Das Frontend wird dabei in das Backend integriert.

In [ ]:
%%bash
yarn install --immutable
yarn tsc
yarn build:backend

## 6. Container-Image erstellen

Das Dockerfile liegt im Backend-Paket. Der Build-Kontext bleibt jedoch das Stammverzeichnis des Backstage-Projekts.

In [ ]:
%%bash
docker image build \
  --file packages/backend/Dockerfile \
  --tag mybackstage:1.0.0 \
  .

## 7. Container starten

Die Umgebungsvariablen werden aus `.env.production` geladen. Backstage ist danach über Port `7007` erreichbar.

In [ ]:
%%bash
docker run \
  --name mybackstage \
  --rm \
  --env-file .env.production \
  --publish 7007:7007 \
  mybackstage:1.0.0

Backstage ist anschliessend unter folgender Adresse erreichbar:

`http://192.168.1.104:7007`